# Lab 02 — Horse or Human: khảo sát dữ liệu
Nguồn: [TFDS](https://www.tensorflow.org/datasets/catalog/horses_or_humans),
[tác giả Laurence Moroney](https://laurencemoroney.com/datasets.html).
Giấy phép dữ liệu: CC BY 2.0 theo tác giả. Đây là ảnh CGI, không phải ảnh chụp thực.
Nhãn: horses = 0; humans = 1. Tập validation của tác giả được TFDS gọi là test;
giữ làm holdout cuối cùng. Thống kê dưới đây chỉ tính từ ảnh thực tế trong workspace.
Chạy `python scripts/download_dataset.py` và `python scripts/prepare_dataset.py` trước.
Không sửa split hoặc augmentation trong notebook này.

In [ ]:
from pathlib import Path
import os, sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'source/config.py').is_file()), None)
assert ROOT is not None, 'Run from inside project'
sys.path.insert(0, str(ROOT))
os.environ.setdefault('MPLCONFIGDIR', str(ROOT / '.cache/matplotlib'))
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display
from scripts.prepare_dataset import audit, prepare
FIGURES = ROOT / 'results/figures'
FIGURES.mkdir(parents=True, exist_ok=True)


In [ ]:
# Re-audit every image and verify frozen splits without rewriting them.
report = prepare(ROOT, check_only=True)
inventory, errors = audit(ROOT)
splits = {s: pd.read_csv(ROOT / f'data/splits/{s}.csv') for s in ('train','val','test')}
print('Total readable raw images:', len(inventory))
print('Corrupt files:', len(errors))
print('Exact duplicate files:', report['duplicate_files'])
display(inventory.groupby('class_name').size().rename('raw_count').to_frame())
counts = pd.DataFrame({s: df.groupby('class_name').size() for s,df in splits.items()}).T
counts['Total'] = counts.sum(axis=1)
display(counts)
print(report['policy'], report['ratios'])
display(inventory.groupby(['width','height','format','mode','channels']).size().rename('count').to_frame())


In [ ]:
# Show samples from train only; evaluation images remain a holdout.
samples = splits['train'].sample(n=16, random_state=42)
fig, axes = plt.subplots(4, 4, figsize=(10,10))
for ax, row in zip(axes.flat, samples.itertuples()):
    with Image.open(ROOT / row.image_path) as im:
        ax.imshow(im.convert('RGB'))
    ax.set_title(f'{row.class_name} ({row.label})')
    ax.axis('off')
fig.tight_layout()
fig.savefig(FIGURES / 'dataset_samples.png', dpi=150)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(6,4))
inventory.groupby('class_name').size().plot.bar(ax=ax, rot=0)
ax.set(title='Raw class distribution', ylabel='Images')
fig.tight_layout(); fig.savefig(FIGURES / 'class_distribution.png', dpi=150)
plt.show()
fig, ax = plt.subplots(figsize=(7,4))
counts[['horses','humans']].plot.bar(ax=ax, rot=0)
ax.set(title='Frozen split distribution', ylabel='Images')
fig.tight_layout(); fig.savefig(FIGURES / 'split_distribution.png', dpi=150)
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10,4))
for ax, dim in zip(axes, ['width','height']):
    inventory[dim].value_counts().sort_index().plot.bar(ax=ax, rot=0)
    ax.set(title=f'Image {dim}', xlabel='Pixels', ylabel='Images')
fig.tight_layout(); fig.savefig(FIGURES / 'image_size_distribution.png', dpi=150)
plt.show()


In [ ]:
# Findings are generated from measured values, not expected catalog counts.
print(f"Measured {len(inventory)} readable images; {len(errors)} corrupt files; "
      f"{report['duplicate_files']} exact duplicate files excluded.")
print('Observed color modes:', inventory['mode'].value_counts().to_dict())
print('Observed resolutions:', inventory.groupby(['width','height']).size().to_dict())
print('Class fractions:', inventory.class_name.value_counts(normalize=True).round(4).to_dict())
print('Hash checks detect identical files, not near-duplicate CGI poses or scenes.')


## Giới hạn và bước tiếp theo
Ảnh CGI có thể khác ảnh chụp thực về nền, ánh sáng và texture (nguồn: tác giả).
Không suy diễn độ chính xác trên ảnh thực từ EDA. SHA-256 không xác nhận độc lập giữa
các nhân vật/cảnh gần giống nhau. Không dùng test để chọn mô hình.
RGB conversion và resize 64×64 sẽ được triển khai ở M1; augmentation chỉ áp dụng train ở M2.
